# Глава 8 — Multi-Agent Systems

Один оркестратор управляет двумя специалистами: **даты** и **арифметика**. Специалист для него — обычный инструмент, функция которого вызывает другой `TinyAgent.run()`.

```text
Пользователь → Оркестратор
                ├── ask_date_agent(question) → Агент дат → today / days_between
                └── ask_math_agent(question) → Математик → add / subtract / multiply
```

Это централизованная многоагентная система. Здесь вызовы последовательные: математик получает число дней, которое сначала вычислил агент дат. Три агента используют одну установленную `gemma4:e4b`, но у каждого свои инструменты, память, planner и траектория. Специализация достигается набором инструментов, а не обучением отдельной модели.

Запустите Jupyter из корня проекта. Внешние сервисы и новые модели не нужны.

In [ ]:
from datetime import date
import math
import re
import socket

from llm import LLM
from multi_agent import create_agent_team
from toolbox import days_between
from illustrated_agents.utils import TrajectoryViewer

socket.setdefaulttimeout(180)
llm = LLM(model="gemma4:e4b", think=True, temperature=0)
team = create_agent_team(llm, max_steps=6)
orchestrator_agent = team.orchestrator_agent
math_agent = team.math_agent
date_agent = team.date_agent

for name, agent in (("orchestrator", orchestrator_agent), ("date", date_agent), ("math", math_agent)):
    print(name, "→", list(agent.tools.registry))

## Как специалист становится инструментом

В `multi_agent.py` функции `ask_math_agent(question: str)` и `ask_date_agent(question: str)` замыкаются на экземпляры специалистов. Они зарегистрированы в `NativeTools` оркестратора.

Модель видит имя, описание и параметр `question`, но не внутреннюю историю специалиста. Вызов проходит через тот же реестр и проверки аргументов, что обычная Python-функция. В ответ оркестратору попадает финальный текст специалиста.

In [ ]:
print(orchestrator_agent.tools.schemas)
assert len({id(a.memory) for a in (orchestrator_agent, date_agent, math_agent)}) == 3
assert len({id(a.trajectory) for a in (orchestrator_agent, date_agent, math_agent)}) == 3

## Задача из главы: накопления по €4 в день

Уточним неоднозначное «до 2030»: считаем от сегодняшней даты **до 2030-12-31, не включая конечную дату**. Значит, число дней равно `end - start`, без прибавления единицы. `today()` использует локальную дату компьютера; результат меняется со временем.

Пример книги получен в мае 2026, поэтому его сумму нельзя считать фиксированным эталоном. Здесь эталон для проверки вычисляется Python по реально полученной дате. Вопрос явно просит делегировать обе части задачи — это проверка механизма делегирования, а не самостоятельного выбора архитектуры.

In [ ]:
END_DATE = "2030-12-31"
STARTED_ON = date.today().isoformat()
TASK = (
    "If I save EUR 4 per day starting today and stopping before 2030-12-31, how much will I save? "
    "Ask the date specialist to get today and compute the date difference with its tools, "
    "then ask the math specialist to multiply that number of days by 4 with its tool. "
    "The end date is excluded: use days_between(today, 2030-12-31) with no extra day. "
    "State the start date, number of days, and total."
)
answer = orchestrator_agent.run(TASK)
print(answer)

## Траектория оркестратора

Здесь видны делегированные вопросы и ответы специалистов. Внутренние вызовы `today`, `days_between`, `multiply` находятся в отдельных траекториях.

In [ ]:
print(orchestrator_agent.trajectory.runs)
TrajectoryViewer(orchestrator_agent.trajectory)

## Что действительно сделал агент дат

In [ ]:
print(date_agent.trajectory.runs)
TrajectoryViewer(date_agent.trajectory)

## Что действительно сделал математик

In [ ]:
print(math_agent.trajectory.runs)
TrajectoryViewer(math_agent.trajectory)

## Проверяем процесс и результат

Одной правильной суммы недостаточно: модель могла угадать её без инструментов. Проверка ниже требует реальных делегирований, чтения текущей даты, вызова `days_between` с нужными датами и умножения полученного числа дней на 4.

Она проверяет выбранный путь решения этого примера. Другой корректный, но иначе устроенный путь может не пройти такую строгую проверку траектории.

In [ ]:
def action_steps(agent):
    return [step for run in agent.trajectory.runs for step in run["steps"] if step.action]

outer = action_steps(orchestrator_agent)
assert [s.action["tool"] for s in outer] == ["ask_date_agent", "ask_math_agent"]
assert len(date_agent.trajectory.runs) == len(math_agent.trajectory.runs) == 1

date_steps = action_steps(date_agent)
assert [s.action["tool"] for s in date_steps] == ["today", "days_between"]
start = date_steps[0].observation
assert start in {STARTED_ON, date.today().isoformat()}
assert date_steps[1].action["kwargs"] == {"a": start, "b": END_DATE}
expected_days = days_between(start, END_DATE)
assert int(date_steps[1].observation) == expected_days

math_steps = action_steps(math_agent)
assert [s.action["tool"] for s in math_steps] == ["multiply"]
factors = sorted(float(math_steps[0].action["kwargs"][key]) for key in ("a", "b"))
assert factors == sorted([float(expected_days), 4.0])
expected_total = expected_days * 4
assert math.isclose(float(math_steps[0].observation), expected_total, rel_tol=0, abs_tol=1e-9)

for agent in (orchestrator_agent, date_agent, math_agent):
    assert agent.trajectory.runs[-1]["steps"][-1].observation is None
assert outer[0].observation == date_agent.trajectory.runs[-1]["steps"][-1].answer
assert outer[1].observation == math_agent.trajectory.runs[-1]["steps"][-1].answer
compact_answer = re.sub(r"[,\s]", "", answer)
assert re.search(rf"(?<![\d.]){expected_total}(?:\.0+)?(?![\d.])", compact_answer)
print(f"Проверено: {start} → {END_DATE}: {expected_days} дней × 4 = {expected_total} EUR")
print("Шаги по агентам:", {name: sum(len(r["steps"]) for r in agent.trajectory.runs)
                            for name, agent in (("orchestrator", orchestrator_agent), ("date", date_agent), ("math", math_agent))})

## Память, ошибки и лимиты

- Повторный вызов той же команды `team` сохраняет истории всех трёх агентов. Новый `create_agent_team(llm)` создаёт полностью свежую команду.
- Для независимых примеров главы 7 используйте `Evaluator(lambda: create_agent_team(llm).orchestrator_agent)` — новый оркестратор вместе с новыми специалистами на каждый пример.
- `max_steps` относится к **каждому** вызову `run()`, включая каждый новый вызов специалиста. Это не общий бюджет. При лимите `S` у этой фиксированной схемы верхняя граница — `S + S²` запросов генерации: до `S` у оркестратора и до `S` внутри каждого делегирования.
- Время работы и число токенов этим лимитом не ограничены. Сводки памяти здесь не используются.
- Ошибка специалиста, пустой ответ или исчерпание его лимита превращаются в ошибочный observation у оркестратора. Он может объяснить ошибку или повторить делегирование в пределах своего лимита. Ошибка не выдаётся за успешный ответ специалиста.
- Завершение оркестратора само по себе не доказывает успех всей задачи — поэтому выше проверяются все три траектории.
- `requires_approval` применяется отдельно к каждому реестру: можно контролировать делегирование у оркестратора и конкретные функции у специалиста. Разрешение на одно не является автоматически разрешением на другое.

## Остальная теория главы

Централизованная схема проста, но зависит от оркестратора. В децентрализованной агенты договариваются напрямую; в иерархической есть несколько уровней управления; в федеративной взаимодействуют системы разных организаций.

CAMEL, MetaGPT, коммуникационные протоколы вроде A2A, социальные симуляции и research-системы обсуждаются в книге как более сложные подходы. В этом проекте они не подключаются: реализован кодовый пример «агенты как инструменты».

Несколько агентов не гарантируют лучшего результата: появляются дополнительные запросы, потери информации при пересказе, ошибки координации и более сложная оценка. Сравнивать такую систему с одним агентом нужно на одинаковых задачах и с учётом затрат.